In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/zmyhome')
START_URL = "https://zmyhome.com/home/list-new-condo?typeAds=sale"
OUTPUT_CSV_FILE = BASE_DIR / 'zmyhome_listing_urls.csv'

TARGET_URL_COUNT = 200
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum performance by blocking heavy resources."""
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)


def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    """Extracts unique project listing URLs from Zmyhome."""
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a[href^="/property/"]'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    # Zmyhome paths start with /property/..., so we attach the base URL
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    all_urls = set()

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)

        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/property/"]')))
        except TimeoutException:
            logging.warning("Timeout waiting for elements to load. Stopping.")
            return

        last_height = driver.execute_script("return document.body.scrollHeight")
        retries = 0

        while len(all_urls) < TARGET_URL_COUNT:
            # ดึงลิงก์จากหน้าจอปัจจุบันและอัปเดตเข้า Set (Set จะตัดลิงก์ซ้ำให้อัตโนมัติ)
            current_links = extract_links(driver, base_url)
            all_urls.update(current_links)

            logging.info(f"Collected {len(all_urls)} / {TARGET_URL_COUNT} URLs...")

            if len(all_urls) >= TARGET_URL_COUNT:
                break

            # เลื่อนจอลงด้านล่างสุดเพื่อกระตุ้นให้เว็บโหลดข้อมูลเพิ่ม (Infinite Scroll)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)  # รอโหลดข้อมูลสักครู่

            # เช็คว่าเลื่อนจอลงได้อีกไหม ถ้าความสูงจอไม่เปลี่ยนแปลว่าถึงสุดทางแล้ว
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                retries += 1
                if retries >= 5:  # ให้โอกาสรอโหลด 5 ครั้ง หากสุดจอจริงให้หยุดการทำงาน
                    logging.info("Reached the bottom of the page or no more items are loading.")
                    break
            else:
                retries = 0
                last_height = new_height

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    # ตัดให้เหลือ 200 URL พอดีเผื่อกรณีเลื่อนครั้งสุดท้ายแล้วได้ทะลุเป้าหมาย
    final_urls = sorted(list(all_urls))[:TARGET_URL_COUNT]

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in final_urls:
            writer.writerow([u])

    logging.info(f"Successfully saved {len(final_urls)} URLs to {OUTPUT_CSV_FILE}")


if __name__ == "__main__":
    main()

2026-03-29 16:36:00,225 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:36:01,224 - INFO - Starting scrape at: https://zmyhome.com/home/list-new-condo?typeAds=sale
2026-03-29 16:36:05,904 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:07,955 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:10,010 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:12,024 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:14,047 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:16,103 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:18,116 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:20,130 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:22,144 - INFO - Collected 12 / 200 URLs...
2026-03-29 16:36:24,158 - INFO - Reached the bottom of the page or no more items are loading.
2026-03-29 16:36:24,305 - INFO - Successfully saved 12 URLs to /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping

In [2]:
import csv
import logging
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/zmyhome')
INPUT_CSV = BASE_DIR / 'zmyhome_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'zmyhome_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-extensions")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    # 1. Listing Title (ชื่อประกาศ และ รหัส)
    try:
        title = driver.find_element(By.TAG_NAME, 'h1').text.strip()
        # ตัดเว้นบรรทัดออกให้เหลือบรรทัดเดียว
        title = " ".join(title.splitlines())
        data.extend(["--- Listing Title ---", title])
    except NoSuchElementException:
        pass

    # 2. Seller Info (ข้อมูลผู้ขาย)
    try:
        seller = driver.find_element(By.CSS_SELECTOR, '.avatar-detail h4').text.strip()
        data.extend(["\n--- Seller ---", seller])
    except NoSuchElementException:
        pass

    # 3. Price (ราคา)
    try:
        price = driver.find_element(By.CSS_SELECTOR, '.info-price__list-right .priceRoom .currency.th').text.strip()
        price_sqm = driver.find_element(By.CSS_SELECTOR, '.info-price__list-right .priceRoom .currency-per-unit').text.strip()
        data.extend(["\n--- Price ---", f"{price} บาท ({price_sqm})"])
    except NoSuchElementException:
        pass

    # 4. Property Info (ข้อมูลห้อง: พื้นที่, จำนวนห้องนอน, ชั้น ฯลฯ)
    try:
        info_items = driver.find_elements(By.CSS_SELECTOR, '.info-room__list .info-room__list--item')
        if info_items:
            data.append("\n--- Property Info ---")
            for item in info_items:
                try:
                    label = item.find_element(By.CSS_SELECTOR, '.small').text.strip()
                    value = item.find_element(By.CSS_SELECTOR, '.label').text.strip()
                    data.append(f"{label}: {value}")
                except NoSuchElementException:
                    continue
    except NoSuchElementException:
        pass

    # 5. Description (ข้อมูลรายละเอียดจากเจ้าของ)
    try:
        desc = driver.find_element(By.ID, 'propertyDetailBox').text.strip()
        if desc:
            data.extend(["\n--- Description ---", desc])
    except NoSuchElementException:
        pass

    # 6. Location & Nearby (ทำเลที่ตั้ง และ สถานที่ใกล้เคียง)
    try:
        data.append("\n--- Location & Nearby ---")
        address = driver.find_element(By.CSS_SELECTOR, '.nearby-place__address').text.strip()
        data.append(f"Address: {address}")

        nearby_items = driver.find_elements(By.CSS_SELECTOR, '#propertyLocationBox .nearby-place__list__item')
        for item in nearby_items:
            try:
                # ดึงหัวข้อหมวดหมู่ เช่น รถไฟฟ้า, ศูนย์การค้า
                category_elements = item.find_elements(By.CSS_SELECTOR, '.small')
                if not category_elements:
                    continue
                category = category_elements[0].text.strip()
                
                # ดึงชื่อสถานที่ใกล้เคียงในหมวดหมู่นั้นๆ
                labels = [lbl.text.strip() for lbl in item.find_elements(By.CSS_SELECTOR, '.label')]
                if category and labels:
                    data.append(f"{category}: {', '.join(labels)}")
            except NoSuchElementException:
                continue
    except NoSuchElementException:
        pass

    return "\n".join(data)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    # อ่าน URL จากไฟล์
    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)  # Skip header
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    # รอให้แท็ก <h1> ของชื่อประกาศแสดงผล
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

2026-03-29 16:39:20,654 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:39:42,903 - INFO - Progress: [10/12] scraped.
2026-03-29 16:39:48,224 - INFO - Scraping completed. Saved to: /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/zmyhome/zmyhome_full_details.csv
